# From GIS data to a controlled district-heating network
## GEOppi live lecture and practical exercise

This notebook introduces GIS inputs, GEOppi modelling functions, load assumptions,
a diameter catalogue and controller settings. The default dataset is
**`exampleNetwork2`**. All modelling results remain in memory; there are no GeoPackage, CSV,
pickle or figure-export commands.

**Learning outcomes**
1. Turn streets, buildings and producer sites into a feed-line pandapipes network.
2. Distinguish annual energy demand, design power and a hydraulic simulation load case.
3. Size radial branches directly and mesh/producer-connection pipes iteratively.
4. Assign each loop its largest selected nominal width and update insulation properties.
5. Build feed and return layers, solve hydraulics and temperatures, and introduce controllers.

**Suggested session:** preparation and GIS (15 min), sizing and loops (25 min), thermal model
(15 min), controllers and discussion (20 min). Run cells in order. Use the existing
`env_geoppi` kernel with the editable GEOppi installation. Restart the kernel before repeating
the complete workflow after changing inputs. Large datasets take longer to build and control.

## 1. Environment and input files
The notebook locates the examples directory through the imported editable package, rather
than using `__file__`, which is unavailable in ordinary notebook cells. All inputs are read
from `geoppi/examples/data/<name>`. Keep the kernel attached to the intended installation.

In [ ]:
### Global plotting settings
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as tck
import matplotlib.patches as patches
import matplotlib.dates as mdates


plt.rcParams["font.family"] = 'Times New Roman'
plt.rcParams["figure.figsize"] = (18.35/2.54, 18.35/2.54 * 0.618) # Default is approx width of elsarticle.cls template with option 3p (two column width) and assumed 10pt fontsize and golden ratio for height
plt.rcParams["figure.facecolor"] = 'white'
plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["axes.labelsize"] = 10
plt.rcParams["axes.facecolor"] = 'white'
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10
plt.rcParams["axes.axisbelow"] = True

plt.rcParams["grid.alpha"] = 0.5
plt.rcParams["grid.linestyle"] = '-'
plt.rcParams["grid.linewidth"] = 1.25

plt.rcParams["legend.fontsize"] = 'small'
plt.rcParams["legend.framealpha"] = 0.9
plt.rcParams["legend.loc"] = 'best'

mpl.rcParams["axes.formatter.use_locale"] = True

In [ ]:
from pathlib import Path
import rasterio
import geopandas as gp
import pandas as pd
import pandapipes as ppi
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from pandapipes.control import run_control
import geoppi
from geoppi.auxFunctions import simultaneity_rh, simultaneity_DHW, _print_info
from geoppi import (
    transfer_LoadPoint_ppi, extract_FluidProperties_ppi,
    get_dict_from_aggregated_groups, implement_controllers,
)


cs = 'EPSG:25832'
name = 'exampleNetwork2'
examples_dir = Path(geoppi.__file__).resolve().parent / 'examples'
flp = examples_dir / 'data' / name
required_files = ['streets.gpkg', 'buildings.gpkg', 'valves.gpkg', 'producers2.gpkg', 'dgm.tif']
missing_files = [filename for filename in required_files if not (flp / filename).is_file()]
if missing_files:
    raise FileNotFoundError(f'Missing example inputs in {flp}: {missing_files}')

In [ ]:
# Read input layers.
pipes = gp.read_file(flp / 'streets.gpkg').to_crs(cs)
pipes['name'] = np.arange(len(pipes))
buildings = gp.read_file(flp / 'buildings.gpkg').to_crs(cs)
valves = gp.read_file(flp / 'valves.gpkg').to_crs(cs)
producers = gp.read_file(flp / 'producers2.gpkg').to_crs(cs)
dgm = rasterio.open(flp / 'dgm.tif')
display(pd.DataFrame({'Layer': ['Streets', 'Buildings', 'Valves', 'Producers'],
                      'Objects': [len(pipes), len(buildings), len(valves), len(producers)]}))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8), constrained_layout=True)
buildings.plot(ax=ax, color="#e5d9e3", edgecolor='white', linewidth=0.3)
pipes.plot(ax=ax, color='#38546b', linewidth=1.5, label='Candidate streets')
producers.plot(ax=ax, color='#c44e30', edgecolor='black', linewidth=0.6)
if not valves.empty:
    valves.plot(ax=ax, color='#e9a23b', markersize=35, label='Valves')
ax.set_title(f'{name}: candidate routes, buildings and producer sites')
ax.set_xlabel('Easting [m]'); ax.set_ylabel('Northing [m]')
plt.show()

*Figure 1 — GIS input overview. Dark lines are candidate routes, grey polygons are
buildings, orange-red polygons are producer sites and yellow points are valves.*

**Pause and predict:** Which streets provide alternative routes between producers? Which
parts look radial? A visually plausible network may still contain a disconnected line.

## 2. From annual demand to separate design powers
We split `demand_use_th` into **81% room heating (RH)** and **19% domestic hot
water (DHW)**. The full-load-hour assumptions are **1500 h for RH** and **500 h for
DHW**. These are example assumptions, not values inferred from the geometry.

$$P_{RH}=0.81E/1500,\qquad P_{DHW}=0.19E/500$$

With annual demand in kWh/a, these calculations produce kW. The undiscounted sum provides
the simulated load case. The two components remain available for branch-specific diversity.
Buildings with missing demand need attention: a missing value is not a measured zero.

In [ ]:
# Define unique ID for identification of buildings
buildings_uniqueID = 'build_ID'

if buildings_uniqueID not in buildings.columns:
    buildings[buildings_uniqueID] = np.arange(len(buildings))

# Define attribute containing heating demand information
heatingDemandAttr = 'demand_use_th'

# Split annual demand (kWh/a) before network creation, as specified for this example.
spaceHeatingPowerAttr = 'design_rh_kW'
hotWaterPowerAttr = 'design_DHW_kW'
consumerPowerAttr = 'design_total_kW'
buildings['demand_rh_kWh'] = buildings[heatingDemandAttr] * 0.81
buildings['demand_DHW_kWh'] = buildings[heatingDemandAttr] * 0.19
# Convert each annual demand with its own full-load hours to design power (kW).
buildings[spaceHeatingPowerAttr] = buildings['demand_rh_kWh'] / 1500.0
buildings[hotWaterPowerAttr] = buildings['demand_DHW_kWh'] / 500.0
buildings[consumerPowerAttr] = buildings[spaceHeatingPowerAttr] + buildings[hotWaterPowerAttr]
# Explicit extras demonstrate transfer of names without the historic 'pth' substring.
buildingAddAttr = ['demand_rh_kWh', 'demand_DHW_kWh', consumerPowerAttr]

In [ ]:
display(pd.DataFrame({
    'Annual demand [MWh/a]': [np.round(buildings['demand_rh_kWh'].sum()/1000,3),
                             np.round(buildings['demand_DHW_kWh'].sum()/1000,3)],
    'Design power [kW]': [np.round(buildings[spaceHeatingPowerAttr].sum(),0),
                                      np.round(buildings[hotWaterPowerAttr].sum(),0)],
}, index=['Room heating', 'Domestic hot water']))

display(pd.Series({'Buildings with unknown annual demand': int(buildings[heatingDemandAttr].isna().sum())}))
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
axes[0].bar(['RH', 'DHW'], [buildings['demand_rh_kWh'].sum()/1000,
                           buildings['demand_DHW_kWh'].sum()/1000], color=['#3978a8', '#df8b38'])
axes[0].set_ylabel('Annual demand [MWh/a]'); axes[0].set_title('Energy split')
axes[1].bar(['RH', 'DHW'], [buildings[spaceHeatingPowerAttr].sum(),
                           buildings[hotWaterPowerAttr].sum()], color=['#3978a8', '#df8b38'])
axes[1].set_ylabel('Design power [kW]'); axes[1].set_title('Power split after full-load-hour conversion')
plt.show()

*Figure 2 — Energy and power are different quantities: the DHW power share is larger
than its annual energy share because the assumed full-load hours are lower.*

**Exercise:** Predict the change in DHW design power if its full-load hours are halved.
Do not change the notebook midway through a run; restart from this section when exploring it.

## 3. Hydraulic design parameters
We use the KMR pipe catalogue, allowed nominal widths, 65/40 °C temperature
pair, 8 bar network pressure, 100 Pa/m pressure-loss target and pipe-length factor of 1.3.
`valves_closed = [-1]` closes all valves for the sizing case. The explicit consumer power
column takes precedence over the legacy `flh` fallback in `transfer_LoadPoint_ppi`.

In [ ]:
### Network type (either "KMR" or "5GDHC")
networkType = 'KMR'

### Set temperatures in the network (°C)
tFeed = 65
tReflux = 40

# Set network pressure at peak load producers (bar)
pNetwork = 8

### Valves
# Selection of valves to be closed ("-1": all closed)
valves_closed = [-1]

# Legacy fallback only; explicit consumer design powers below take precedence
flh = (1700)

# Define attribute name containing information about thermal power (kW) of producers in producer models
thermalPowerAttr = 'Pth_kW'

# Define attribute name containing infotmation about distribution ('VE') and house connection pipes ('HA')
connectionTypeAttr = 'connectionType'

# Define naming convention for base load producers in heat_consumer component DataFrame
baseLoadProdNaming = 'prod'

# Target specific pressure loss in pipes (Pa/m, usually 60-100 Pa/m)
dp_spec = 100

# Selection of subset for nominal widths for pipes
subsetDistribution = [32, 40, 50, 65, 80, 100, 125, 150, 200, 225, 250, 300, 350]
subsetConnection = [25, 32, 40, 50, 65, 80, 100, 125, 150, 200, 225, 300]

# Desired insulation for pipes
insulationType = 'std'

# Medium in the network
fluid = 'water'

# Factor for elongation of pipes to consider influence of armatures etc. on pressure loss
factor_l = 1.3

# Pressure loss calculation model for pandapipes
friction_model = 'swamee-jain'

# Selection if real heights at junctions shall be considered
respect_height = False

# Access network type-dependent data for pipes
dictNominalInnerDiameter, dictWallRoughness = geoppi.load_hydraulic_pipe_characteristics(networkType)

# Keep numerical warnings visible during the practical exercise.

In [ ]:
counts = np.arange(1, 101)
fig, ax = plt.subplots(figsize=(9, 3.5), constrained_layout=True)
ax.plot(counts, [simultaneity_rh(n) for n in counts], label='Room heating')
ax.plot(counts, [simultaneity_DHW(n) for n in counts], label='Domestic hot water')
ax.set(xlabel='Downstream buildings', ylabel='Simultaneity factor', title='Simultaneity factors')
ax.legend(); ax.grid(alpha=0.2)
plt.show()

*Figure 3 — The plotted factors are evaluated from the installed functions, including
any limits implemented there. They are not independently reimplemented in this notebook.*

## 4. House connections and the feed-line network
`create_connection_lines` links building polygons to candidate streets and splits intersecting
lines. Its default maximum connection distance is **100 m**. A building outside that search
radius can remain unconnected even when the street network itself is connected.

`create_ppi_network_from_gdf` then creates junctions, pipes, valves, sinks and producers.
The separate design-power names and additional building columns are transferred into the
network; subsequent pipe sizing needs only `netFeed`, not the building layer.

In [ ]:
# Optional, if house connection lines have not yet been created:
pipes = geoppi.create_connection_lines(
    lines = pipes,
    polys = buildings,
    dist_connection_points = 2,
    max_dist_connection_lines = 100,
)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8), constrained_layout=True)
buildings.plot(ax=ax, color="#EE07EE", edgecolor='white', linewidth=0.2)
pipes.plot(ax=ax, color='#2c7487', linewidth=0.8)
producers.plot(ax=ax, color='#c44e30')
ax.set_title('Candidate network after house-connection generation')
ax.set_xlabel('Easting [m]'); ax.set_ylabel('Northing [m]')
plt.show()

*Figure 4 — House connections attach buildings to the street-derived network. Zoom into the displayed plot when discussing individual connections.*

In [ ]:
netFeed = geoppi.create_ppi_network_from_gdf(
    pipes = pipes,
    buildings = buildings,
    valves = valves,
    producers = producers,
    rasterHeight = dgm,
    buildings_uniqueID = 'build_ID',
    heatingDemandAttr = heatingDemandAttr,
    spaceHeatingPowerAttr = spaceHeatingPowerAttr,
    hotWaterPowerAttr = hotWaterPowerAttr,
    buildingAddAttr = buildingAddAttr,
    networkName = 'exampleNet',
    networkFluid = fluid,
    producerTypeDict = {'Type':{'peak':'peak','base':'base'}},
    producerAddAttr = ['Pth_kW', 'dp_bar_internal'],
    modelling_type = 'feed_line'
)

In [ ]:
# Reconcile building coverage before interpreting any sizing result.
unconnected = buildings.loc[~buildings[buildings_uniqueID].isin(netFeed.sink[buildings_uniqueID])]
display(pd.Series({
    'Building records': len(buildings), 'Created sinks': len(netFeed.sink),
    'Unconnected buildings': len(unconnected),
    'Input known demand [MWh/a]': buildings[heatingDemandAttr].sum()/1000,
    'Connected known demand [MWh/a]': netFeed.sink[heatingDemandAttr].sum()/1000,
    'Excluded known demand [MWh/a]': unconnected[heatingDemandAttr].sum()/1000,
}))
if not unconnected.empty:
    display(unconnected[[buildings_uniqueID, heatingDemandAttr]])
display(netFeed.sink[[buildings_uniqueID, spaceHeatingPowerAttr, hotWaterPowerAttr, consumerPowerAttr]].head())

**Checkpoint:** A connected street network does not guarantee that every building has
a house connection. Discuss omitted demand before proceeding. The table deliberately keeps
missing-demand records visible rather than silently interpreting them as supplied loads.

## 5. Apply the simulation load case
`transfer_LoadPoint_ppi` sets consumer and producer powers/mass flows, supply temperatures and
pressure boundary conditions. It does **not** choose pipe diameters. Here it uses
`design_total_kW`, not annual demand divided by the fallback full-load hours.

The basic conversion is $\dot m=P\,1000/(c_p\Delta T)$.
This globally defined simulation case remains distinct from local diversified design loads.

In [ ]:
netFeed = transfer_LoadPoint_ppi(net = netFeed,
    tFeed = tFeed + 273.15,
    tReflux = tReflux + 273.15,
    pNetwork = pNetwork,
    flh = flh,
    heatingDemandAttr = heatingDemandAttr,
    thermalPowerAttr = thermalPowerAttr,
    thermalPowerAttrConsumers = consumerPowerAttr,
    baseLoadProdNaming = baseLoadProdNaming,
    check_for_mass_flow_exceeding = True
)

## 6. Size pipes: radial branches and iterative regions
`hydraulicDimensioningNetwork_singleLoadPoint` combines two approaches:

- **Radial branches:** bridges outside loops and outside connections between producers.
  Downstream building counts determine the separate diversity factors and a suitable catalogue
  diameter is selected directly.
- **Other pipes:** hydraulic simulation and diameter updates determine the sizes. Connections
  between producers are not automatically radial, even if they lie outside a loop.

The stored pipe direction is unchanged. `design_power_kW` retains diversified radial loads;
other pipe powers use the final absolute mass flow and temperature difference. Consequently,
`design_Pa_per_m` and simulated `Pa_per_m` describe different cases on diversified branches.

In [ ]:
cp_fluid, rho_fluid, nu_fluid, g = extract_FluidProperties_ppi(net = netFeed, t = 0.5 * (tFeed + tReflux))

# Adapt user defined options of the network to hydraulic and thermal calculation mode
ppi.set_user_pf_options(net = netFeed, mode="hydraulics", friction_model = friction_model, quit_on_inconsistency_connectivity=True, reset = True)

netFeed = geoppi.hydraulicDimensioningNetwork_singleLoadPoint(
    net = netFeed,
    networkType = networkType,
    dpSpec = dp_spec,
    thresh_percentage = [0, 0.2],
    dictNominalInnerDiameter = {'distributionPipes':dictNominalInnerDiameter, 'connectionPipes':dictNominalInnerDiameter},
    dictWallRoughness = dictWallRoughness,
    subsetDistribution = subsetDistribution,
    subsetConnection = subsetConnection,
    connectionTypeAttr = connectionTypeAttr,
    elongationFactorPipes = factor_l,
    nominalWidthAttr = 'nominalWidth',
    closedValvesIdx = valves_closed,
    frictionModel = friction_model,
    respectHeight = False,
    fluidProperties = {'rho':rho_fluid, 'cp':cp_fluid, 'nu':nu_fluid},
    greaterWidthFinalStep={'distributionPipes':0, 'connectionPipes':0},
    # Consumer powers and their column names are already stored in netFeed.
    deltaT=tFeed - tReflux,
    simultaneitySpaceHeating=simultaneity_rh,
    simultaneityHotWater=simultaneity_DHW,
    # design_* pipe columns describe local diversified sizing, not res_pipe loads.
)


### Assign final heat transfer coefficient
netFeed.pipe = geoppi.assign_insulation_type(
    df_pipes = netFeed.pipe, 
    colNameNominalWidth = 'nominalWidth', 
    network_type = networkType, 
    insulation_type = insulationType if networkType == 'KMR' else None
)

netFeed = geoppi.update_ppi_results(
    net = netFeed,
    user_pf_options={'mode':'hydraulics', 'friction_model':'swamee-jain'},
    respectHeight = False,
    fluidProperties = {'rho':rho_fluid, 'cp':cp_fluid, 'nu':nu_fluid},
    elongationFactorPipes = 1.3,
    resultAttributes={ 'pipe': ['Pa_per_m', 'mdot_from_kg_per_s'],'junction': ['t_k', 'p_bar'] }
)

In [ ]:
from shapely import box

gdf = gp.GeoDataFrame(netFeed.pipe.copy(), geometry='geometry', crs=cs)
distribution = gdf.loc[gdf.in_service & gdf[connectionTypeAttr].eq('distribution')]
fig, axes = plt.subplots(1, 3, figsize=(17, 6), constrained_layout=True)
distribution.plot(ax=axes[0], column='nominalWidth', cmap='Paired', categorical=True,
                  legend=True, linewidth=2, legend_kwds={'title': 'DN', 'loc':'upper center',
                  'bbox_to_anchor':(0.5,-0.02), 'ncol':4, 'fontsize':9})
boxp = gp.GeoDataFrame(geometry=[box(distribution.total_bounds[0],distribution.total_bounds[1],distribution.total_bounds[2],distribution.total_bounds[3])], crs=distribution.crs)
boxp.plot(ax=axes[0], color="#8b8a8a", zorder=0)

if distribution['loop'].notna().any():
    distribution.plot(ax=axes[1], column='loop', cmap='Paired', categorical=True,
                      legend=True, linewidth=2, missing_kwds={'color':'#c8c8c8', 'label':'No loop'},
                      legend_kwds={'loc':'upper center', 'bbox_to_anchor':(0.5,-0.02), 'ncol':3, 'fontsize':9})
    boxp.plot(ax=axes[1], color="#8b8a8a", zorder=0)
else:
    distribution.plot(ax=axes[1], color='#c8c8c8', linewidth=2)
distribution.plot(ax=axes[2], column='Pa_per_m', cmap='magma', vmin=0, vmax=200,
                  legend=True, linewidth=2, legend_kwds={'label':'Pa/m'})
boxp.plot(ax=axes[2], color="#8b8a8a", zorder=0)

for ax, title in zip(axes, ['Selected nominal widths', 'Stored loop affiliation', 'Simulated pressure loss']):
    ax.set_title(title); ax.set_axis_off()
plt.show()
display(netFeed.pipe.groupby('is_radial')['design_power_kW'].agg(['count', 'min', 'max']))

*Figure 5 — Nominal width, loop affiliation
and specific pressure loss. Colour limits of 0–200 Pa/m support comparison between runs.
The power table reports a range, not a sum: summing powers along a branch would count the
same downstream buildings repeatedly.*

**Discussion:** Why might a radial pipe's simulated pressure loss exceed the design target
even when its diversified local design loss satisfies it?

## 7. Give each stored loop its largest nominal width
`get_dict_from_aggregated_groups(..., func='max')` groups pipes by `loop` and builds a mapping
from the maximum nominal width to affected pipe indices. `assign_nominal_widths_manually`
updates nominal width, inner diameter and roughness together. Pipes without loop affiliation
are not part of that grouping. This uses the stored loop labels; it is not a
search over every possible cycle in the graph.

We retain `netFeed` for comparison and work on `netFeed_dimensioned`. Insulation is assigned
again because the heat-transfer coefficient depends on diameter.

In [ ]:
_, dictNWs = get_dict_from_aggregated_groups(
    df = netFeed.pipe,
    groupCol = 'loop',
    val = 'nominalWidth',
    func = 'max'
)

netFeed_dimensioned = geoppi.assign_nominal_widths_manually(
    net = netFeed,
    dictNW = dictNWs,
    dictNominalInnerDiameter = dictNominalInnerDiameter,
    dictWallRoughness = dictWallRoughness,
    nominalWidthAttr = 'nominalWidth'
)

### Assign final heat transfer coefficient
netFeed_dimensioned.pipe = geoppi.assign_insulation_type(
    df_pipes = netFeed_dimensioned.pipe, 
    colNameNominalWidth = 'nominalWidth', 
    network_type = networkType, 
    insulation_type = insulationType if networkType == 'KMR' else None
)

In [ ]:
# Recalculate with the same helper before comparing hydraulic results after the overwrite.
netFeed_dimensioned = geoppi.update_ppi_results(
    net=netFeed_dimensioned,
    user_pf_options={'mode':'hydraulics', 'friction_model':friction_model},
    respectHeight=False, fluidProperties={'rho':rho_fluid, 'cp':cp_fluid, 'nu':nu_fluid},
    elongationFactorPipes=factor_l,
    resultAttributes={'pipe':['Pa_per_m', 'mdot_from_kg_per_s'], 'junction':['t_k', 'p_bar']},
)
comparison = pd.DataFrame({'Loop': netFeed.pipe['loop'], 'Before DN': netFeed.pipe.nominalWidth,
                           'After DN': netFeed_dimensioned.pipe.nominalWidth})
display(comparison.loc[comparison.Loop.notna()].groupby('Loop').agg(
    before_min=('Before DN','min'), before_max=('Before DN','max'),
    after_min=('After DN','min'), after_max=('After DN','max')))
before = gp.GeoDataFrame(netFeed.pipe[netFeed.pipe[connectionTypeAttr]=="distribution"], geometry='geometry', crs=cs)
after = gp.GeoDataFrame(netFeed.pipe[netFeed.pipe[connectionTypeAttr]=="distribution"], geometry='geometry', crs=cs)

boxp = gp.GeoDataFrame(geometry=[box(before.total_bounds[0],before.total_bounds[1],before.total_bounds[2],before.total_bounds[3])], crs=before.crs)

fig, axes = plt.subplots(1, 2, figsize=(13, 6), constrained_layout=True)
vmin, vmax = before.nominalWidth.min(), after.nominalWidth.max()
for ax, frame, title in zip(axes, [before, after], ['Before loop overwrite', 'After loop overwrite']):
    frame.plot(ax=ax, column='nominalWidth', cmap='viridis', vmin=vmin, vmax=vmax,
               linewidth=1.6, legend=True, legend_kwds={'label':'Nominal width (DN)'})
    boxp.plot(ax=ax, color="#8b8a8a", zorder=0)
    ax.set_title(title); ax.set_axis_off()
plt.show()

*Figure 6 — Identical colour limits make the diameter overwrite visible. Increasing
diameters changes the hydraulic resistance, so the comparison uses refreshed results.
The original sizing `design_*` columns are not a fresh optimization after this manual edit.*

## 8. Build feed and return layers; solve hydraulics and heat transport
`feed_and_reflux_line_circPumps` builds the combined network: feed-line pipes are mirrored into a return
layer, sinks are represented by heat consumers between the layers, and producers are
represented by circulation pumps. Base-load producers use mass-flow pumps with flow controllers.

`mode='sequential'` calculates hydraulics followed by heat transport. Reapply the load case
to the combined model before reading its results. The external pipe temperature is 280 K.
The `alpha_w_per_m2k` and `u_w_per_m2k` aliases carry the same heat-transfer coefficient.

In [ ]:
netFeedReflux = geoppi.create_ppi_network_from_gdf(
    existingNetworkFeedLine = netFeed_dimensioned,
    buildings = buildings,
    heatingDemandAttr = heatingDemandAttr,
    producers = producers,
    networkName = 'exampleNet',
    networkFluid = 'water',
    producerTypeDict = {'Type':{'peak':'peak','base':'base'}},
    producerAddAttr = ['Pth_kW', 'dp_bar_internal'], # Necessary thermal power attribute to write into pandapipes tables
    modelling_type = 'feed_and_reflux_line_circPumps'
)

# Set user-defined pipeflow option for sequential calculation!
ppi.set_user_pf_options(net = netFeedReflux, reset = True, mode='sequential', friction_model = friction_model, quit_on_inconsistency_connectivity=True)

netFeedReflux = transfer_LoadPoint_ppi(
    net = netFeedReflux,
    tFeed = tFeed + 273.15,
    tReflux = tReflux + 273.15,
    text_pipes = 280,
    pNetwork = pNetwork,
    flh = flh,
    heatingDemandAttr = heatingDemandAttr,
    thermalPowerAttr = thermalPowerAttr,
    thermalPowerAttrConsumers = consumerPowerAttr,
    baseLoadProdNaming = baseLoadProdNaming
)

netFeedReflux = geoppi.update_ppi_results(
    net = netFeedReflux,
    respectHeight=False,
    fluidProperties={'cp':cp_fluid, 'rho':rho_fluid, 'nu':nu_fluid},
    elongationFactorPipes=factor_l,
    resultAttributes = {'pipe':['Pa_per_m', 'mdot_from_kg_per_s'], 'junction':['t_k', 'p_bar'], 'heat_consumer':['t_from_k', 't_to_k']}
)

In [ ]:
from shapely.geometry import box

# Keep a baseline for comparison after the controllers have run.
uncontrolled_pressure = netFeedReflux.res_junction['p_bar'].copy()
uncontrolled_return = netFeedReflux.res_heat_consumer['t_to_k'].copy() - 273.15

# Store the arithmetic mean of both pipe-end temperatures on the network itself.
netFeedReflux.pipe['mean_temperature_degC'] = (
    netFeedReflux.res_pipe['t_from_k'] + netFeedReflux.res_pipe['t_to_k']
) / 2 - 273.15
thermal_map = gp.GeoDataFrame(netFeedReflux.pipe.copy(), geometry='geometry', crs=cs)
distribution = thermal_map.loc[
    thermal_map.in_service & thermal_map[connectionTypeAttr].eq('distribution')
]

# Match consumer inlet temperatures to building IDs, independently of row order.
active_consumers = netFeedReflux.heat_consumer.loc[netFeedReflux.heat_consumer.in_service]
supply_temperatures = pd.Series(
    netFeedReflux.res_heat_consumer.loc[active_consumers.index, 't_from_k'].to_numpy() - 273.15,
    index=active_consumers[buildings_uniqueID],
)
building_temperatures = buildings.copy()
building_temperatures['supply_temperature_degC'] = (
    building_temperatures[buildings_uniqueID].map(supply_temperatures)
)

# Keep the grey background box and include the complete building footprints.
plot_bounds = gp.GeoSeries(pd.concat(
    [distribution.geometry, building_temperatures.geometry], ignore_index=True
), crs=cs).total_bounds
background = gp.GeoDataFrame(geometry=[box(*plot_bounds)], crs=cs)
fig, axes = plt.subplots(1, 2, figsize=(13, 6), constrained_layout=True)
for ax, layer, title in zip(
    axes, ['feedLine', 'refluxLine'], ['Mean feed-pipe temperatures', 'Mean return-pipe temperatures']
):
    background.plot(ax=ax, color='#8b8a8a', zorder=0)
    # Both panels show building supply temperatures on the same temperature scale.
    building_temperatures.plot(
        ax=ax, column='supply_temperature_degC', cmap='coolwarm',
        vmin=tReflux-5, vmax=tFeed, zorder=1,
        missing_kwds={'color': '#c8c8c8'},
    )
    selected = distribution.loc[distribution.Layer.eq(layer)]
    selected.plot(
        ax=ax, column='mean_temperature_degC', cmap='coolwarm',
        vmin=tReflux-5, vmax=tFeed, linewidth=1.5, zorder=2,
        legend=True, legend_kwds={'label': 'Temperature [°C]'},
    )
    ax.set_title(title)
    ax.set_axis_off()
plt.show()

*Figure 7 — Distribution-pipe temperatures before controller action: each line shows the
arithmetic mean of its two end temperatures, stored as `pipe.mean_temperature_degC`.
House connections are omitted for clarity. Building footprints show their heat consumer's
supply temperature in both panels; buildings without an active consumer result appear light
grey. Pipes and buildings share the same colour scale, over a grey background box.
The thermal model includes heat transport and heat losses.*

## 9. Spatial interpretation: accumulated annual demand
The next section reproduces the `sum_attrs_to_closest_supplier` analysis on feed
pipes and open feed valves. Edge length is the path weight. The result is accumulated annual
energy demand, **not** a hydraulic flow solution and **not** a diversified design power.
In meshes, shortest-path allocation should not be confused with physical flow splitting.

In [ ]:
import pandas as pd

# Prepare dataframe to convert
## Keep opened valves
weightAttr = 'length_km'
colsToKeep = ['from_junction', 'to_junction', weightAttr, 'geometry']

if hasattr(netFeedReflux, 'valve'):

    DF_edges = pd.concat((
        netFeedReflux.pipe[netFeedReflux.pipe['Layer'] == 'feedLine'][[n for n in colsToKeep if n in netFeedReflux.pipe.columns]], 
        netFeedReflux.valve[(netFeedReflux.valve['Layer'] == 'feedLine') & (netFeedReflux.valve['opened'] == True)][[n for n in colsToKeep if n in netFeedReflux.valve.columns]]
        ), axis = 0)
    
else:
    DF_edges = netFeedReflux.pipe[netFeedReflux.pipe['Layer'] == 'feedLine'][[n for n in colsToKeep if n in netFeedReflux.pipe.columns]]

DF_edges.loc[DF_edges[weightAttr].isna(), weightAttr] = 0
DF_edges = gp.GeoDataFrame(DF_edges, geometry = "geometry").set_crs(cs)


DF_out = geoppi.sum_attrs_to_closest_supplier(
    lines = DF_edges,
    buildings = gp.GeoDataFrame(netFeedReflux.heat_consumer, geometry = "geometry").set_crs(cs),
    producers = gp.GeoDataFrame(netFeedReflux.junction[netFeedReflux.junction.index.isin(list(netFeedReflux.circ_pump_pressure["flow_junction"]) + list(netFeedReflux.circ_pump_mass["flow_junction"]))], geometry = "geometry").set_crs(cs),
    building_attrs = ['demand_use_th'],
    weight = "length_km",
    building_id = "build_ID", 
)

In [ ]:
gdf = gp.GeoDataFrame(DF_out, geometry='geometry', crs=cs)
fig, ax = plt.subplots(figsize=(11, 8), constrained_layout=True)
gdf.plot(ax=ax, column='sum_demand_use_th', cmap='magma', vmin=0,
         vmax=max(1, gdf['sum_demand_use_th'].max()*0.75), linewidth=2, legend=True,
         legend_kwds={'label':'Accumulated annual heat demand [kWh/a]'})
ax.set_title('Annual demand allocated towards the closest heat supplier')
ax.set_axis_off()
plt.show()

*Figure 8 — Shortest-path demand aggregation. The upper colour limit
is 75% of the maximum value to reveal differences along smaller branches.*

## 10. Add and run the controllers
The controller configuration below sets the operating targets. `drop_all=True` replaces
existing controllers, preventing duplicate registration when this cell is repeated.

| Controller configuration | Target | Level / order |
|---|---|---|
| `TRefluxHeatConsumerCtrlDict` | Consumer return temperature: 40 °C; minimum flow 0.015 kg/s; minimum ΔT 3 K | −1 / 0 |
| `dpminCtrlDict` | Minimum consumer differential pressure: 1.2 bar | 5 / 0 |
| `pminCtrlDict` | Minimum junction pressure: 1.5 bar | 5 / 1 |
| `PthLimitedCtrlDict` | Producer power targets from `Pth_kW` | 5 / 5 |

Lower level and order values are scheduled earlier. The specified tolerances and bounds
matter: controller targets are not a guarantee that every displayed value is exactly equal
to the target. The producer configuration assumes the same base-load arrangement as the
example; inspect it before reusing these settings with another dataset.

In [ ]:
# Define control targets
circ_pump_pressure_idx = 0

dpmin_target = 1.2 # bar
pmin_target = 1.5 # bar

ppi.set_user_pf_options(net = netFeedReflux, mode="sequential", friction_model = friction_model, quit_on_inconsistency_connectivity=True, reset = True)

# Create controllers 
netFeedReflux = geoppi.implement_controllers(
    net = netFeedReflux,
    drop_all = True,
    pminCtrlDict = {
        'create':True, 
        'pmin_target':pmin_target, 
        'circ_pump_pressure_idx':circ_pump_pressure_idx,
        'abs_tol':0.1,
        'order':1,
        'level':5
        },
    dpminCtrlDict = { # Lowest control order in level 5 = highest priority in level 5
        'create':True, 
        'dpmin_target':dpmin_target, 
        'circ_pump_pressure_idx':circ_pump_pressure_idx,
        'index':1,
        'abs_tol':0.1,
        'order':0,
        'level':5
        },
    TRefluxHeatConsumerCtrlDict = { # Lowest control level = highest priority
        'create':True,
        'T_target':tReflux + 273.15,
        'min_mdot':0.015,
        'min_dT':3,
        'abs_tol':0.1,
        'order':0,
        'level':-1
        },
    PthLimitedCtrlDict = {
        'create':True,
        'circ_pump_mass_idxs':netFeedReflux.circ_pump_mass.index, 
        'circ_pump_pressure_index':circ_pump_pressure_idx, 
        'flow_controller_idxs':netFeedReflux.flow_control.index,
        'Pth_target_kW':netFeedReflux.circ_pump_mass['Pth_kW'].values, 
        'priority_list':[0],
        'abs_tol':5000,
        'order':5,
        'level':5
        }
    )

In [ ]:
display(netFeedReflux.controller[['in_service', 'order', 'level']])
# This can take longer than one pipeflow solve because the controllers iterate.
run_control(net=netFeedReflux, max_iter=50)

## 11. Check the achieved operating point
We check the smallest consumer differential pressure, the smallest junction
pressure and the distribution of consumer return temperatures. We show these as a compact
table and compare before/after distributions. Values are read from the latest controlled
simulation, not from the earlier sizing results.

In [ ]:
netFeedReflux.res_heat_consumer['dp_bar'] = (
    netFeedReflux.res_heat_consumer['p_from_bar'] - netFeedReflux.res_heat_consumer['p_to_bar'])
dpmin_reached = netFeedReflux.res_heat_consumer['dp_bar'].min()
idx_dpmin = netFeedReflux.res_heat_consumer['dp_bar'].idxmin()
buildIDD = netFeedReflux.heat_consumer.at[idx_dpmin, 'build_ID']
pmin_reached = netFeedReflux.res_junction['p_bar'].min()
display(pd.DataFrame({'Target [bar]':[dpmin_target,pmin_target],
                      'Achieved [bar]':[dpmin_reached,pmin_reached]},
                     index=['Minimum consumer differential pressure','Minimum junction pressure']))
display(pd.Series({'Critical heat-consumer index':idx_dpmin, 'Critical building ID':buildIDD}))
controlled_return = netFeedReflux.res_heat_consumer['t_to_k'] - 273.15
fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
axes[0].hist([uncontrolled_pressure.dropna(), netFeedReflux.res_junction.p_bar.dropna()],
             bins=20, label=['Before controls','After controls'], alpha=0.8)
axes[0].axvline(pmin_target, color='black', linestyle='--', label='Minimum target')
axes[0].set(xlabel='Junction pressure [bar]', ylabel='Junctions', title='Pressure distribution')
axes[0].legend()
axes[1].hist([uncontrolled_return.dropna(), controlled_return.dropna()], bins=20,
             label=['Before controls','After controls'], alpha=0.8)
axes[1].axvline(tReflux, color='black', linestyle='--', label='Return-temperature target')
axes[1].set(xlabel='Consumer return temperature [°C]', ylabel='Consumers', title='Return-temperature distribution')
axes[1].legend()
plt.show()
# Inspect narrow bins around the return-temperature target.
return_bins = pd.cut(netFeedReflux.res_heat_consumer['t_to_k'],
                    np.arange(tReflux+273.15-2, tReflux+273.15+2, 0.25))
display(return_bins.value_counts(normalize=True).sort_index())
display(pd.Series({'Consumers outside the displayed bins or without a result': int(return_bins.isna().sum())}))

*Figure 9 — Before/after operating-point comparison. The narrow temperature-bin table
normalizes over values inside its bins; the separately reported excluded count prevents an
apparently good histogram from hiding out-of-range consumers.*

## Practical exercises and closing questions
1. Change `dp_spec`, rerun from network creation and compare selected DN and pressure losses.
2. Change one simultaneity callable and explain which sizing values change, and why the
   globally imposed sink loads do not automatically acquire the same branch-wise diversity.
3. Compare minimum/maximum loop DN before and after the manual overwrite.
4. Compare the controlled minimum pressure and return-temperature distributions with their
   targets and tolerances. Identify the limiting consumer by its building ID.
5. Explain why a missing house connection, an isolated street and an inactive consumer are
   three different diagnoses.

**Troubleshooting:** resolve disconnected geometry before modelling; reconcile unconnected
building demand; check the permitted diameter catalogue if no radial size meets the target;
check producer indices, bounds and the simulation mode if controls do not converge.
No input geometries are repaired automatically here.

**Objects available for discussion:** `buildings`, `netFeed`, `netFeed_dimensioned`,
`netFeedReflux` and `DF_out`. They remain in the kernel only; no datasets are exported.

In [ ]:
# Release the input raster handle after the exercise. Reopen it by rerunning section 1.
dgm.close()